# Extract Edge and HOG Features for IPL Grid Cells

This notebook starts from `Dataset_Annotations.csv` and creates a cell-level feature CSV with exactly these base fields:

```text
Image File Name, Train Or Test, cell_row, cell_col, label,
f_edge_density, f_canny_count, f_sobel_mean, f_sobel_std,
f_laplacian_var, f_contour_count
```

It then appends HOG feature columns after them:

```text
f_hog_000, f_hog_001, ..., f_hog_035
```

The output rows are ordered by image first, then by cell position inside that image: `c01` to `c64`.

The notebook does **not** read a separate `TARGET_CSV` / `Edge_features.csv` file. It builds the row-per-cell table directly from the annotation CSV.


## 1. Imports

The feature extraction uses OpenCV and NumPy. HOG is implemented manually from Sobel magnitude and orientation, so `scikit-image` is not required.


In [3]:
# Run this only if the libraries are not installed in your environment.
# %pip install pandas numpy opencv-python huggingface_hub

from pathlib import Path
from collections import defaultdict
import re

import cv2
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download


## 2. Configuration

Change the annotation file name or output CSV name here if needed.

`cell_row` and `cell_col` are **0-based** values from `0` to `7`. This makes image slicing simple:

```text
x_min = cell_col * 100
y_min = cell_row * 75
```


In [4]:
# Input annotation CSV with wide labels c01 ... c64.
ANNOTATION_CSV = "Dataset_Annotations.csv"

# Output CSV with edge + HOG features.
OUTPUT_CSV = "Dataset_Features_Edge.csv"

# Hugging Face dataset containing the images.
HF_REPO_ID = "goyaljai/IPL-Player-Detection-IITB-PML"
HF_REPO_TYPE = "dataset"

# Project-required image size.
IMAGE_WIDTH = 800
IMAGE_HEIGHT = 600

# Project grid size.
GRID_ROWS = 8
GRID_COLS = 8

CELL_WIDTH = IMAGE_WIDTH // GRID_COLS      # 100 pixels
CELL_HEIGHT = IMAGE_HEIGHT // GRID_ROWS    # 75 pixels

# Basic edge feature columns.
BASIC_EDGE_FEATURE_COLUMNS = [
    "f_edge_density",
    "f_canny_count",
    "f_sobel_mean",
    "f_sobel_std",
    "f_laplacian_var",
    "f_contour_count",
]

# HOG settings.
# Each 100x75 cell is split into 2x2 regions.
# Each region gets a 9-bin orientation histogram.
# Total HOG features per cell = 2 * 2 * 9 = 36.
HOG_PATCH_ROWS = 2
HOG_PATCH_COLS = 2
HOG_BINS = 9

HOG_FEATURE_COLUMNS = [
    f"f_hog_{i:03d}"
    for i in range(HOG_PATCH_ROWS * HOG_PATCH_COLS * HOG_BINS)
]

FEATURE_COLUMNS = BASIC_EDGE_FEATURE_COLUMNS + HOG_FEATURE_COLUMNS

OUTPUT_COLUMNS = [
    "Image File Name",
    "Train Or Test",
    "cell_row",
    "cell_col",
    "label",
    *FEATURE_COLUMNS,
]

print("Basic edge feature columns:", len(BASIC_EDGE_FEATURE_COLUMNS))
print("HOG feature columns:", len(HOG_FEATURE_COLUMNS))
print("Total feature columns:", len(FEATURE_COLUMNS))
print("Total output columns:", len(OUTPUT_COLUMNS))


Basic edge feature columns: 6
HOG feature columns: 36
Total feature columns: 42
Total output columns: 47


## 3. Read annotations and create one row per image-cell

The annotation file has one row per image and columns `c01` to `c64`. For model training, we convert it into one row per image-cell.

The output is ordered **image-first**, then cell order within that image:

```text
img_1.jpg, c01
img_1.jpg, c02
...
img_1.jpg, c64
img_2.jpg, c01
img_2.jpg, c02
...
```

The `label` column is the target team ID for that cell.


In [5]:
annotations_df = pd.read_csv(ANNOTATION_CSV)

print("Loaded annotation CSV")
print("Shape:", annotations_df.shape)
print("Columns:", annotations_df.columns.tolist())

required_annotation_cols = ["Image File Name", "Train Or Test"]
missing_required_cols = [c for c in required_annotation_cols if c not in annotations_df.columns]
if missing_required_cols:
    raise ValueError(f"Missing required columns in annotation CSV: {missing_required_cols}")

# Find c01 ... c64 columns.
cell_label_cols = [
    c for c in annotations_df.columns
    if re.fullmatch(r"c\d{2}", str(c))
]

expected_cell_cols = [f"c{i:02d}" for i in range(1, 65)]
missing_cell_cols = [c for c in expected_cell_cols if c not in cell_label_cols]
if missing_cell_cols:
    raise ValueError(f"Missing expected cell label columns: {missing_cell_cols}")

# Keep c01 ... c64 in correct order.
cell_label_cols = expected_cell_cols

# Preserve the original image order from the annotation CSV.
# This prevents the output from being arranged as:
# c01 for all images, then c02 for all images, etc.
annotations_df = annotations_df.copy()
annotations_df["__image_order"] = np.arange(len(annotations_df))

# Wide format -> long cell-level format.
df = annotations_df.melt(
    id_vars=["__image_order", "Image File Name", "Train Or Test"],
    value_vars=cell_label_cols,
    var_name="cell_id_temp",
    value_name="label",
)

# Convert c01 -> 1, c64 -> 64.
df["__cell_num"] = df["cell_id_temp"].str[1:].astype(int)

# 0-based row and column values: 0 to 7.
df["cell_row"] = ((df["__cell_num"] - 1) // GRID_COLS).astype(int)
df["cell_col"] = ((df["__cell_num"] - 1) % GRID_COLS).astype(int)

# Reorder rows so the CSV is image-major:
# img_1 c01, img_1 c02, ..., img_1 c64, img_2 c01, ...
df = df.sort_values(["__image_order", "__cell_num"]).reset_index(drop=True)

# Label should be numeric team ID 0-10.
df["label"] = df["label"].astype(int)

# Add empty feature columns.
for col in FEATURE_COLUMNS:
    df[col] = np.nan

# Keep only the fields requested, with HOG features after existing edge features.
# Temporary ordering columns are intentionally removed here.
df = df[OUTPUT_COLUMNS]

print("Created cell-level feature table")
print("Shape:", df.shape)
print("First 12 rows should all be from the first image, with cell_col increasing from 0 to 7 and then row 1.")
df.head(12)


Loaded annotation CSV
Shape: (1013, 67)
Columns: ['Image File Name', 'Train Or Test', 'count', 'c01', 'c02', 'c03', 'c04', 'c05', 'c06', 'c07', 'c08', 'c09', 'c10', 'c11', 'c12', 'c13', 'c14', 'c15', 'c16', 'c17', 'c18', 'c19', 'c20', 'c21', 'c22', 'c23', 'c24', 'c25', 'c26', 'c27', 'c28', 'c29', 'c30', 'c31', 'c32', 'c33', 'c34', 'c35', 'c36', 'c37', 'c38', 'c39', 'c40', 'c41', 'c42', 'c43', 'c44', 'c45', 'c46', 'c47', 'c48', 'c49', 'c50', 'c51', 'c52', 'c53', 'c54', 'c55', 'c56', 'c57', 'c58', 'c59', 'c60', 'c61', 'c62', 'c63', 'c64']
Created cell-level feature table
Shape: (64832, 47)
First 12 rows should all be from the first image, with cell_col increasing from 0 to 7 and then row 1.


,Image File Name,Train Or Test,cell_row,cell_col,label,f_edge_density,f_canny_count,f_sobel_mean,f_sobel_std,f_laplacian_var,...,f_hog_026,f_hog_027,f_hog_028,f_hog_029,f_hog_030,f_hog_031,f_hog_032,f_hog_033,f_hog_034,f_hog_035
0,img_1.jpg,Train,0,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,img_1.jpg,Train,0,1,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,img_1.jpg,Train,0,2,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,img_1.jpg,Train,0,3,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,img_1.jpg,Train,0,4,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,img_1.jpg,Train,0,5,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,img_1.jpg,Train,0,6,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,img_1.jpg,Train,0,7,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,img_1.jpg,Train,1,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,img_1.jpg,Train,1,1,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Download the image dataset and build an image index

The notebook uses `Image File Name` to locate images inside the dataset.


In [6]:
dataset_dir = Path(
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE
    )
)

print("Dataset directory:", dataset_dir)

all_image_paths = (
    list(dataset_dir.rglob("*.jpg"))
    + list(dataset_dir.rglob("*.jpeg"))
    + list(dataset_dir.rglob("*.png"))
)

image_paths_by_name = defaultdict(list)
for path in all_image_paths:
    image_paths_by_name[path.name].append(path)

print("Total image files found:", len(all_image_paths))
print("Unique image file names found:", len(image_paths_by_name))

duplicate_names = {name: paths for name, paths in image_paths_by_name.items() if len(paths) > 1}
print("Duplicate image file names:", len(duplicate_names))

if duplicate_names:
    first_name = next(iter(duplicate_names))
    print("Example duplicate name:", first_name)
    print(duplicate_names[first_name])


Fetching ... files: 0it [00:00, ?it/s]

Dataset directory: C:\Users\Rishabh Sharda\.cache\huggingface\hub\datasets--goyaljai--IPL-Player-Detection-IITB-PML\snapshots\3188c4d51921e60ac3da5a10c6dca43136c81199
Total image files found: 1013
Unique image file names found: 1013
Duplicate image file names: 0


## 5. Helper function to locate an image


In [7]:
def find_image_path(image_file_name):
    """Return the local image path for an image file name from the CSV."""

    matches = image_paths_by_name.get(str(image_file_name), [])

    if not matches:
        return None

    return matches[0]


## 6. Compute edge and gradient maps for one full image

For efficiency, we compute Canny, Sobel, Laplacian, and gradient orientation once per full image. Then each cell is sliced from those full-image maps.


In [8]:
def compute_edge_maps(image_bgr):
    """
    Compute edge-related maps for the full image.

    Input:
        image_bgr: OpenCV image in BGR format, resized to 800x600.

    Output:
        Dictionary containing full-image edge and gradient maps.
    """

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Canny edge map.
    canny = cv2.Canny(blurred, threshold1=50, threshold2=150)

    # Sobel gradients.
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # Gradient magnitude and unsigned orientation for HOG.
    sobel_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)
    sobel_angle = np.rad2deg(np.arctan2(sobel_y, sobel_x)) % 180

    # Laplacian for high-frequency detail / sharpness.
    laplacian = cv2.Laplacian(blurred, cv2.CV_64F)

    return {
        "canny": canny,
        "sobel_mag": sobel_mag,
        "sobel_angle": sobel_angle,
        "laplacian": laplacian,
    }


## 7. Extract edge and HOG features for one cell

The code uses `cell_row` and `cell_col` to calculate the crop boundaries internally. The output CSV does not need `x_min`, `y_min`, `x_max`, or `y_max` columns.


In [9]:
def extract_hog_features(sobel_mag_cell, sobel_angle_cell):
    """
    Extract simple HOG-style features from one image cell.

    Each cell is divided into HOG_PATCH_ROWS x HOG_PATCH_COLS patches.
    For each patch, a HOG_BINS-bin orientation histogram is computed,
    weighted by Sobel gradient magnitude.
    """

    hog_values = []

    row_edges = np.linspace(0, sobel_mag_cell.shape[0], HOG_PATCH_ROWS + 1, dtype=int)
    col_edges = np.linspace(0, sobel_mag_cell.shape[1], HOG_PATCH_COLS + 1, dtype=int)

    for r in range(HOG_PATCH_ROWS):
        for c in range(HOG_PATCH_COLS):
            r0, r1 = row_edges[r], row_edges[r + 1]
            c0, c1 = col_edges[c], col_edges[c + 1]

            mag_patch = sobel_mag_cell[r0:r1, c0:c1].ravel()
            angle_patch = sobel_angle_cell[r0:r1, c0:c1].ravel()

            hist, _ = np.histogram(
                angle_patch,
                bins=HOG_BINS,
                range=(0, 180),
                weights=mag_patch,
            )

            # Normalize each patch histogram to reduce scale differences.
            hist_sum = hist.sum()
            if hist_sum > 0:
                hist = hist / hist_sum

            hog_values.extend(hist.astype(float).tolist())

    return {
        col: float(value)
        for col, value in zip(HOG_FEATURE_COLUMNS, hog_values)
    }


def extract_cell_features(edge_maps, cell_row, cell_col):
    """
    Extract edge and HOG features for one grid cell.

    cell_row and cell_col are 0-based values from 0 to 7.
    """

    x_min = int(cell_col) * CELL_WIDTH
    y_min = int(cell_row) * CELL_HEIGHT
    x_max = x_min + CELL_WIDTH
    y_max = y_min + CELL_HEIGHT

    canny_cell = edge_maps["canny"][y_min:y_max, x_min:x_max]
    sobel_cell = edge_maps["sobel_mag"][y_min:y_max, x_min:x_max]
    sobel_angle_cell = edge_maps["sobel_angle"][y_min:y_max, x_min:x_max]
    laplacian_cell = edge_maps["laplacian"][y_min:y_max, x_min:x_max]

    cell_area = canny_cell.shape[0] * canny_cell.shape[1]
    canny_count = int(np.count_nonzero(canny_cell))
    edge_density = canny_count / cell_area if cell_area > 0 else 0.0

    sobel_mean = float(np.mean(sobel_cell))
    sobel_std = float(np.std(sobel_cell))
    laplacian_var = float(np.var(laplacian_cell))

    contours, _ = cv2.findContours(
        canny_cell,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )
    contour_count = int(len(contours))

    features = {
        "f_edge_density": edge_density,
        "f_canny_count": canny_count,
        "f_sobel_mean": sobel_mean,
        "f_sobel_std": sobel_std,
        "f_laplacian_var": laplacian_var,
        "f_contour_count": contour_count,
    }

    features.update(
        extract_hog_features(
            sobel_mag_cell=sobel_cell,
            sobel_angle_cell=sobel_angle_cell,
        )
    )

    return features


## 8. Populate edge and HOG features

The loop groups by `Image File Name`, processes each image once, and fills the feature columns for all 64 cells.


In [10]:
missing_images = []
unreadable_images = []
processed_images = 0
processed_rows = 0

for image_file_name, group in df.groupby("Image File Name", sort=False):
    image_path = find_image_path(image_file_name)

    if image_path is None:
        missing_images.append(image_file_name)
        continue

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        unreadable_images.append(str(image_path))
        continue

    # Ensure the image matches the required 800x600 size.
    image_bgr = cv2.resize(image_bgr, (IMAGE_WIDTH, IMAGE_HEIGHT))

    edge_maps = compute_edge_maps(image_bgr)

    for idx in group.index:
        cell_row = int(df.at[idx, "cell_row"])
        cell_col = int(df.at[idx, "cell_col"])

        features = extract_cell_features(
            edge_maps=edge_maps,
            cell_row=cell_row,
            cell_col=cell_col,
        )

        for col, value in features.items():
            df.at[idx, col] = value

        processed_rows += 1

    processed_images += 1

print("Processed images:", processed_images)
print("Processed cell rows:", processed_rows)
print("Missing images:", len(missing_images))
print("Unreadable images:", len(unreadable_images))

if missing_images:
    print("First missing images:", missing_images[:10])

if unreadable_images:
    print("First unreadable image paths:", unreadable_images[:10])


Processed images: 1013
Processed cell rows: 64832
Missing images: 0
Unreadable images: 0


## 9. Validate and preview the output


In [11]:
print("Missing values in feature columns:")
print(df[FEATURE_COLUMNS].isna().sum())

print("Preview with basic edge features and first 12 HOG features:")
df[[
    "Image File Name",
    "Train Or Test",
    "cell_row",
    "cell_col",
    "label",
    *BASIC_EDGE_FEATURE_COLUMNS,
    *HOG_FEATURE_COLUMNS[:12],
]].head(20)


Missing values in feature columns:
f_edge_density     0
f_canny_count      0
f_sobel_mean       0
f_sobel_std        0
f_laplacian_var    0
f_contour_count    0
f_hog_000          0
f_hog_001          0
f_hog_002          0
f_hog_003          0
f_hog_004          0
f_hog_005          0
f_hog_006          0
f_hog_007          0
f_hog_008          0
f_hog_009          0
f_hog_010          0
f_hog_011          0
f_hog_012          0
f_hog_013          0
f_hog_014          0
f_hog_015          0
f_hog_016          0
f_hog_017          0
f_hog_018          0
f_hog_019          0
f_hog_020          0
f_hog_021          0
f_hog_022          0
f_hog_023          0
f_hog_024          0
f_hog_025          0
f_hog_026          0
f_hog_027          0
f_hog_028          0
f_hog_029          0
f_hog_030          0
f_hog_031          0
f_hog_032          0
f_hog_033          0
f_hog_034          0
f_hog_035          0
dtype: int64
Preview with basic edge features and first 12 HOG features:


,Image File Name,Train Or Test,cell_row,cell_col,label,f_edge_density,f_canny_count,f_sobel_mean,f_sobel_std,f_laplacian_var,...,f_hog_002,f_hog_003,f_hog_004,f_hog_005,f_hog_006,f_hog_007,f_hog_008,f_hog_009,f_hog_010,f_hog_011
0,img_1.jpg,Train,0,0,0,0.060667,455.0,57.836068,40.741251,11.664933,...,0.072833,0.043753,0.067528,0.084570,0.132065,0.142878,0.187145,0.142846,0.062125,0.061526
1,img_1.jpg,Train,0,1,0,0.018933,142.0,37.648206,25.513181,5.905847,...,0.024586,0.012568,0.010174,0.024608,0.057327,0.209775,0.280265,0.151471,0.070954,0.093477
2,img_1.jpg,Train,0,2,0,0.040667,305.0,49.568082,37.760162,8.247790,...,0.130465,0.089093,0.084200,0.079406,0.165225,0.107943,0.144389,0.043536,0.027543,0.050264
3,img_1.jpg,Train,0,3,0,0.040400,303.0,39.715569,28.080953,6.231177,...,0.109148,0.046818,0.065017,0.088624,0.062218,0.157927,0.203600,0.184302,0.079123,0.053963
4,img_1.jpg,Train,0,4,0,0.016533,124.0,40.257663,27.651509,6.034347,...,0.062561,0.089995,0.050541,0.089775,0.173309,0.206138,0.146319,0.073179,0.089092,0.188655
5,img_1.jpg,Train,0,5,0,0.019867,149.0,36.037875,29.168826,5.672387,...,0.090969,0.068591,0.078893,0.052608,0.126161,0.183889,0.107296,0.310587,0.073516,0.062710
6,img_1.jpg,Train,0,6,0,0.031867,239.0,45.861416,31.002229,7.671646,...,0.056838,0.074264,0.240461,0.147327,0.088816,0.144132,0.105791,0.228228,0.109952,0.062891
7,img_1.jpg,Train,0,7,0,0.014133,106.0,37.181765,25.130286,5.802400,...,0.134388,0.113125,0.074781,0.051555,0.088276,0.061248,0.125609,0.242904,0.095420,0.085565
8,img_1.jpg,Train,1,0,0,0.075467,566.0,56.095497,44.671924,19.252837,...,0.065918,0.110639,0.201281,0.123368,0.195823,0.139765,0.064986,0.176564,0.128930,0.068963
9,img_1.jpg,Train,1,1,0,0.036000,270.0,44.724366,37.643692,14.066134,...,0.037634,0.058984,0.089051,0.065321,0.053900,0.079919,0.218850,0.112188,0.050018,0.049931


## 10. Save the populated CSV

The output contains only the requested base fields plus HOG columns appended after the existing edge feature columns.


In [12]:
df = df[OUTPUT_COLUMNS]
df.to_csv(OUTPUT_CSV, index=False)

print("Saved populated CSV to:", OUTPUT_CSV)
print("Final shape:", df.shape)
print("Final columns:")
print(df.columns.tolist())

print("\nOrder check: first 20 rows")
print(df[["Image File Name", "cell_row", "cell_col", "label"]].head(20))


Saved populated CSV to: Dataset_Features_Edge.csv
Final shape: (64832, 47)
Final columns:
['Image File Name', 'Train Or Test', 'cell_row', 'cell_col', 'label', 'f_edge_density', 'f_canny_count', 'f_sobel_mean', 'f_sobel_std', 'f_laplacian_var', 'f_contour_count', 'f_hog_000', 'f_hog_001', 'f_hog_002', 'f_hog_003', 'f_hog_004', 'f_hog_005', 'f_hog_006', 'f_hog_007', 'f_hog_008', 'f_hog_009', 'f_hog_010', 'f_hog_011', 'f_hog_012', 'f_hog_013', 'f_hog_014', 'f_hog_015', 'f_hog_016', 'f_hog_017', 'f_hog_018', 'f_hog_019', 'f_hog_020', 'f_hog_021', 'f_hog_022', 'f_hog_023', 'f_hog_024', 'f_hog_025', 'f_hog_026', 'f_hog_027', 'f_hog_028', 'f_hog_029', 'f_hog_030', 'f_hog_031', 'f_hog_032', 'f_hog_033', 'f_hog_034', 'f_hog_035']

Order check: first 20 rows
   Image File Name  cell_row  cell_col  label
0        img_1.jpg         0         0      0
1        img_1.jpg         0         1      0
2        img_1.jpg         0         2      0
3        img_1.jpg         0         3      0
4        i